# MCQ to open-ended question answering
This notebook is for inspecting the MCQ datasets and seeing if they are "trivially" transferable to open-ended question answering tasks where an llm-as-judge is used to evaluate the correctness of the answers.

In [ ]:
import os
import yaml
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import json
from typing import Union, Callable
import time
import datetime

import datasets
from datasets import load_dataset
from openai import OpenAI

In [ ]:
# Setup llm filtering judge client
if os.path.exists("configs/acdc/default.yaml"):
    with open("configs/acdc/default.yaml", "r") as f:
        acd_config = yaml.safe_load(f)
    model_url = f"http://{acd_config['vllm_server_host']}:{acd_config['vllm_server_port']}/v1"
    model_name = acd_config["scientist_model"]
    if model_name.startswith("vllm/"):
        model_name = "/".join(model_name.split("/")[1:])
else:
    # hardcode the url
    model_url = "http://<your IP address>:<Your port>/v1"
    model_name = "Qwen/Qwen2.5-72B-Instruct"

client = OpenAI(api_key="empty", base_url=model_url)

In [ ]:
JUDGE_SYSTEM_PROMPT = """
You are a professional educator. Your job is it to evaluate whether a question is unambiguous and can be answered without the multiple choice options. You need to determine whether it is clear what the question is asking.

You will be given the parsed question that you need to evaluate.

A valid question here means:
- The standalone question can be answered without the multiple choice options.
- It is clear what the question is asking.
- Even if a question is posed as a text continuaiton task, if the continuation can be generated without the context of multiple choice options, then it is valid.
- If the question contains anything along the lines of "Which of the following...", then the question is not valid.

Respond precisely in the following format:

THOUGHT:
<THOUGHT>

DECISION:
<DECISION>

In <THOUGHT>, briefly reason about the question and whether it can be answered without the multiple choice answers.

In <DECISION>, provide your answer as either "Yes" or "No".
"""

JUDGE_USER_PROMPT = """
Question:
{question}
"""


In [ ]:
# Helpers for getting filtering judge decisions
itoa_mapping = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E",
    5: "F",
    6: "G",
    7: "H",
    8: "I",
    9: "J",
    10: "K",
    11: "L",
    12: "M",
    13: "N",
    14: "O",
    15: "P",
    16: "Q",
    17: "R",
    18: "S",
    19: "T",
    20: "U",
    21: "V",
    22: "W",
    23: "X",
    24: "Y",
    25: "Z",
}

def extract_judge_decision(response: str) -> bool:
    """
    Extract the judge's decision from the response and return it as a boolean.
    """
    try:
        decision = response.split("DECISION:")[1].strip()
        if "Yes" in decision:
            return True
        elif "No" in decision:
            return False
        else:
            # If no decision is found, then return False
            return False
    except Exception as e:
        print(f"Error extracting judge decision: {e}\nResponse: {response}")
        return False

def ask_judge(
    client: OpenAI,
    model_name: str,
    question: str,
    choices: list[str],
    target: Union[int, str],
    subject: str,
    logging_dict: dict = None,
    id: int = None,
) -> bool:
    """
    Ask a judge llm to evaluate whether a question that was originally a multiple choice question is valid as an open-ended question.

    """

    choices_str = "\n".join([f"{itoa_mapping[i]}. {choice}" for i, choice in enumerate(choices)])
    if isinstance(target, int):
        target_str = choices[target]
    else:
        target_str = target
    prompt = JUDGE_USER_PROMPT.format(
        question=question,
        choices=choices_str,
        target=target_str
    )

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
    )

    judge_response = response.choices[0].message.content
    judge_decision = extract_judge_decision(judge_response)

    if logging_dict is not None:
        logging_dict[id] = {
            "question": question,
            "choices": choices,
            "target": target,
            "subject": subject,
            "judge_response": judge_response,
            "is_valid": judge_decision,
        }

    return judge_decision

In [ ]:
def process_dataset(
    dataset: Union[datasets.Dataset, dict[str, datasets.Dataset]],
    output_dir: str,
    process_item_fn: Callable,
    max_workers: int = 64,
):
    # Add sample details in place into the answer_dict
    answer_dict = {}
    if isinstance(dataset, dict):
        for subject in tqdm(dataset.keys()):
            ds = dataset[subject]
            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                # Create a dictionary of futures
                future_to_item = {
                    executor.submit(process_item_fn, item, i, answer_dict, subject): i 
                    for i, item in enumerate(ds)
                }
                
                # Process results as they complete with progress bar
                for future in tqdm(as_completed(future_to_item), total=len(future_to_item)):
                    try:
                        future.result()
                    except Exception as e:
                        print(f"Error processing item {future_to_item[future]}: {e}")
    else:
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            # Create a dictionary of futures
            future_to_item = {
                executor.submit(process_item_fn, item, i, answer_dict): i 
                for i, item in enumerate(dataset)
            }

            # Process results as they complete with progress bar
            for future in tqdm(as_completed(future_to_item), total=len(future_to_item)):
                try:
                    future.result()
                except Exception as e:
                    print(f"Error processing item {future_to_item[future]}: {e}")

    # sort by id
    answer_dict = dict(sorted(answer_dict.items(), key=lambda x: x[0]))

    valid_samples = [item for item in answer_dict.values() if item["is_valid"]]
    invalid_samples = [item for item in answer_dict.values() if not item["is_valid"]]

    print(f"Valid samples: {len(valid_samples)}")
    print(f"Invalid samples: {len(invalid_samples)}")

    # save the valid and invalid samples
    with open(f"{output_dir}/valid_samples.json", "w") as f:
        json.dump(valid_samples, f)

    with open(f"{output_dir}/invalid_samples.json", "w") as f:
        json.dump(invalid_samples, f)

    # Get all subjects
    subjects = list(set([item["subject"] for item in valid_samples + invalid_samples]))

    # Count the number of valid and invald samples per subject
    subject_valid_counts = {subject: 0 for subject in subjects}
    subject_invalid_counts = {subject: 0 for subject in subjects}

    for item in valid_samples:
        subject = item["subject"]
        subject_valid_counts[subject] = subject_valid_counts.get(subject, 0) + 1

    for item in invalid_samples:
        subject = item["subject"]
        subject_invalid_counts[subject] = subject_invalid_counts.get(subject, 0) + 1

    # sort the subjects by the number of valid samples
    sorted_subjects = sorted(subject_valid_counts.keys(), key=lambda x: subject_valid_counts[x], reverse=True)

    # percentages of valid samples per subject
    subject_valid_percentages = {subject: subject_valid_counts[subject] / (subject_valid_counts[subject] + subject_invalid_counts[subject]) for subject in sorted_subjects}
    # Sort the dictionary by the values (decreasing)
    subject_valid_percentages = dict(sorted(subject_valid_percentages.items(), key=lambda x: x[1], reverse=True))

    # percentages of invalid samples per subject
    subject_invalid_percentages = {subject: subject_invalid_counts[subject] / (subject_valid_counts[subject] + subject_invalid_counts[subject]) for subject in sorted_subjects}
    # Sort the dictionary by the values (decreasing)
    subject_invalid_percentages = dict(sorted(subject_invalid_percentages.items(), key=lambda x: x[1], reverse=True))

    # save the percentages
    with open(f"{output_dir}/subject_valid_percentages.json", "w") as f:
        json.dump(subject_valid_percentages, f)

    with open(f"{output_dir}/subject_invalid_percentages.json", "w") as f:
        json.dump(subject_invalid_percentages, f)

# MMLU

In [ ]:
mmlu_dataset = load_dataset("cais/mmlu", "all")
ds = mmlu_dataset["test"]

In [ ]:
date_now = datetime.datetime.now().strftime("%Y_%m_%d")
time_now = datetime.datetime.now().strftime("%H_%M_%S")

output_dir = f"dataset_filtering/mmlu/{date_now}/{time_now}"
os.makedirs(output_dir, exist_ok=True)

# iterate over the dataset
def process_item(item, i, answer_dict):
    question = item["question"]
    choices = item["choices"]
    target = item["answer"]
    subject = item["subject"]
    return ask_judge(
        client=client,
        model_name=model_name,
        question=question,
        choices=choices,
        target=target,
        subject=subject,
        logging_dict=answer_dict,
        id=i,
    )

process_dataset(
    dataset=mmlu_dataset,
    output_dir=output_dir,
    process_item_fn=process_item,
)

**Prompt v0** \
Valid samples: 13756 \
Invalid samples: 286

**Prompt v1** \
Valid samples: 9616 \
Invalid samples: 4426

**Prompt v2** \
Valid samples: 6619 \
Invalid samples: 7423

# Big Bench Hard

Dataset has only `input` and `target` fields.

Not all tasks are MCQ tasks.
Those that are have the options directly in the `input` field.

The options are formatted as follows:
```
<question>
Options:
(A) <option_a>
(B) <option_b>
...
```

The target is the letter of the correct option.

We can easily extract the options (and the target) from the `input` field and remove them from the `input` field.

After that, we can ask the judge to evaluate whether the question is valid as an open-ended question.

In [ ]:
dataset_path = "SaylorTwift/bbh"
subset_to_dataset_mapping = {}
for task in tqdm(datasets.get_dataset_infos(dataset_path).keys()):
    ds = load_dataset(dataset_path, task, cache_dir="./dataset_cache")["test"]
    subset_to_dataset_mapping[task] = ds

In [ ]:
failed_samples = {}

date_now = datetime.datetime.now().strftime("%Y_%m_%d")
time_now = datetime.datetime.now().strftime("%H_%M_%S")

output_dir = f"dataset_filtering/bbh/{date_now}/{time_now}"
os.makedirs(output_dir, exist_ok=True)

def extract_choices(input_text):
    # find the line that starts with "Options:"
    options = input_text.split("Options:")[1].strip()
    if options:
        # extract the options after "Options:"
        options = options.split("\n")
        final_options = []
        for option in options:
            if "Yes" in option:
                final_options.append("Yes")
            elif "No" in option:
                final_options.append("No")
            else:
                final_options.append(option.strip())
        return final_options
    else:
        return []

# iterate over the dataset
def process_item(item, i, answer_dict, subject):
    try:
        input_text = item["input"]
        target = item["target"]

        id = f"{subject}_{i}"
        
        # extract the choices and the target
        if "Options:" in input_text:
            choices = extract_choices(input_text)
            if not isinstance(target, str): # When target is 'Yes' or 'No'
                target = choices[target]
            # remove the options from the input
            question = input_text.split("Options:")[0].strip()
        else:
            choices = []
            question = input_text

        judge_response = ask_judge(
            client=client,
            model_name=model_name,
            question=question,
            choices=choices,
            target=target,
            subject=subject,
            logging_dict=answer_dict,
            id=id,
        )
        return judge_response
    except Exception as e:
        id = f"{subject}_{i}"
        print(f"Error processing item {id}: {e}")
        failed_samples[id] = {
            "input": item["input"],
            "target": item["target"],
            "subject": subject,
            "error": str(e),
        }
        return None

process_dataset(
    dataset=subset_to_dataset_mapping,
    output_dir=output_dir,
    process_item_fn=process_item
)

**Prompt v2** \
Valid samples: 5381 \
Invalid samples: 1380

# MMLU Pro

Dataset contains the following structure:
```
{
    "question_id": int,
    "question": str,
    "options": list[str],
    "answer": str,
    "answer_index": int,
    "category": str,
    "src": str,
}
```

In [ ]:
# Load the dataset
dataset_path = "TIGER-Lab/MMLU-Pro"
ds_test = load_dataset(dataset_path)["test"]
# validation set is used for in context examples. There are 5 per subject.
ds_val = load_dataset(dataset_path)["validation"]
ds = {
    "test": ds_test,
    "validation": ds_val,
}

In [ ]:
# process the dataset
failed_samples = {}

date_now = datetime.datetime.now().strftime("%Y_%m_%d")
time_now = datetime.datetime.now().strftime("%H_%M_%S")

output_dir = f"dataset_filtering/mmlu_pro/{date_now}/{time_now}"
os.makedirs(output_dir, exist_ok=True)

def process_item(item, i, answer_dict, split, **kwargs):
    question = item["question"]
    choices = item["options"]
    target = item["answer_index"]
    subject = item["category"]
    id = item["question_id"]
    id = f"{split}_{id}"
    return ask_judge(
        client=client,
        model_name=model_name,
        question=question,
        choices=choices,
        target=target,
        subject=subject,
        logging_dict=answer_dict,
        id=id,
    )

process_dataset(
    dataset=ds,
    output_dir=output_dir,
    process_item_fn=process_item,
)

**Prompt v2** \
Valid samples: 8599 \
Invalid samples: 3433

# GPQA

Diamond subset has only 198 samples. \
Main subset has 448 samples.

In [ ]:
# Load the dataset
dataset_path= "Idavidrein/gpqa"
ds_main = load_dataset(dataset_path, "gpqa_main")["train"]
ds_diamond = load_dataset(dataset_path, "gpqa_diamond")["train"]

In [ ]:
# process the dataset

date_now = datetime.datetime.now().strftime("%Y_%m_%d")
time_now = datetime.datetime.now().strftime("%H_%M_%S")

def process_item(item, i, answer_dict):
    question = item["Question"]
    choices = [
        item["Correct Answer"],
        item["Incorrect Answer 1"],
        item["Incorrect Answer 2"],
        item["Incorrect Answer 3"],
    ]
    target = item["Correct Answer"]
    subject = item["Subdomain"]
    id = f"{subject}_{i}"
    return ask_judge(
        client=client,
        model_name=model_name,
        question=question,
        choices=choices,
        target=target,
        subject=subject,    
        logging_dict=answer_dict,
        id=id,
    )

# gpqa main
output_dir = f"dataset_filtering/gpqa_main/{date_now}/{time_now}"
os.makedirs(output_dir, exist_ok=True)

process_dataset(
    dataset=ds_main,
    output_dir=output_dir,
    process_item_fn=process_item,
)

# gpqa diamond
output_dir = f"dataset_filtering/gpqa_diamond/{date_now}/{time_now}"
os.makedirs(output_dir, exist_ok=True)

process_dataset(
    dataset=ds_diamond,
    output_dir=output_dir,
    process_item_fn=process_item,
)



**Prompt v2**

**gpqa_main** \
Valid samples: 298 \
Invalid samples: 150

**gpqa_diamond** \
Valid samples: 137 \
Invalid samples: 61